In [22]:
import os
import time
import math
import numpy as np
from PIL import Image
from skimage.feature import hog
from skimage.transform import resize
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

In [23]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [24]:
patch_size = (48, 48)
stride = 24
hog_params = dict(
    pixels_per_cell=(8,8),
    cells_per_block=(2,2),
    orientations=9,
    block_norm='L2-Hys',
    feature_vector=True
)

## DATA LOADING

In [30]:
def extract_patches_arr(img_arr, patch_size=(48,48), stride=24):
    """
    Returns (N_patches, ph, pw) and list of positions (x,y,w,h).
    If no patch is possible (very small image), returns a single resized patch of the whole image.
    """
    H, W = img_arr.shape
    ph, pw = patch_size
    patches = []
    positions = []
    if H < 1 or W < 1:
        return np.zeros((0,ph,pw), dtype=img_arr.dtype), []
    for y in range(0, max(1, H - ph + 1), stride):
        for x in range(0, max(1, W - pw + 1), stride):
            patch = img_arr[y:y+ph, x:x+pw]
            if patch.shape != (ph, pw):
                patch = resize(patch, patch_size, preserve_range=True).astype(np.uint8)
            patches.append(patch)
            positions.append((x, y, pw, ph))
    if len(patches) == 0:
        # fallback: center crop resized to patch_size
        cpatch = resize(img_arr, patch_size, preserve_range=True).astype(np.uint8)
        patches = [cpatch]
        positions = [(0,0,pw,ph)]
    return np.stack(patches, axis=0), positions

class PatchSequenceDataset(Dataset):
    def __init__(self, root_dir, hog_params, patch_size=(48,48), stride=24):
        self.paths = []
        for cls in os.listdir(root_dir):
            cls_path = os.path.join(root_dir, cls)
            if not os.path.isdir(cls_path): continue
            for f in os.listdir(cls_path):
                if f.lower().endswith((".png",".jpg",".jpeg")):
                    self.paths.append(os.path.join(cls_path, f))
        self.hog_params = hog_params
        self.patch_size = patch_size
        self.stride = stride

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        p = self.paths[idx]
        img = Image.open(p).convert("L")
        arr = np.array(img)
        patches, positions = extract_patches_arr(arr, patch_size=self.patch_size, stride=self.stride)
        feats = [hog(patch, **self.hog_params) for patch in patches]
        feats = np.stack(feats, axis=0).astype(np.float32)   # (L, D)

        # --- Label generation fix ---
        H, W = arr.shape
        cx, cy = W / 2.0, H / 2.0
        max_dist = np.sqrt(cx**2 + cy**2)  # farthest possible distance (corner)
        labels = np.zeros(len(positions), dtype=np.float32)
        
        for i, (x, y, w, h) in enumerate(positions):
            px = x + w/2
            py = y + h/2
            dist = np.sqrt((px - cx)**2 + (py - cy)**2)
            norm_dist = dist / max_dist
            # Positive if within 0.3 of image radius
            if norm_dist < 0.3:
                labels[i] = 1.0
        
            
        return torch.from_numpy(feats), torch.from_numpy(labels)

In [26]:
# Cell 2: collate, dataset, split, loaders, quick sanity checks

def collate_pad(batch):
    """
    batch: list of tuples (feat_tensor (L,D), label_tensor (L,))
    returns: feats_padded (B, Lmax, D), labels_padded (B, Lmax), mask (B, Lmax) bool (True = valid)
    """
    feats, labels = zip(*batch)
    lengths = [f.shape[0] for f in feats]
    feats_padded = pad_sequence(feats, batch_first=True)    # (B, Lmax, D)
    labels_padded = pad_sequence(labels, batch_first=True)  # (B, Lmax)
    maxL = feats_padded.shape[1]
    mask = (torch.arange(maxL).unsqueeze(0) < torch.tensor(lengths).unsqueeze(1))
    mask = mask.to(torch.bool)
    return feats_padded, labels_padded, mask

# ---- configure paths and load dataset ----
data_dir = "Data"   # change if necessary
dataset = PatchSequenceDataset(data_dir, hog_params, patch_size, stride)
print("Loaded dataset with", len(dataset), "images")

# deterministic split
SEED = 42
torch.manual_seed(SEED)
train_ratio = 0.8
n = len(dataset)
train_size = int(train_ratio * n)
val_size = n - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# DataLoaders
BATCH_SIZE = 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_pad, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_pad, pin_memory=True)

print(f"Train size: {len(train_dataset)} | Val size: {len(val_dataset)}")

# quick token-level label distribution (on train)
token_counts = {0:0, 1:0}
for i in range(min(200, len(train_dataset))):   # sample up to 200 items for speed
    feats, labels = train_dataset[i]
    vals = labels.numpy().ravel().astype(int)
    token_counts[0] += int((vals==0).sum())
    token_counts[1] += int((vals==1).sum())
print("Token-level label counts (train sample up to 200 images):", token_counts)

# sample batch shapes sanity check
batch = next(iter(train_loader))
print("sample batch shapes: feats", batch[0].shape, "labels", batch[1].shape, "mask", batch[2].shape, "mask.sum:", batch[2].sum().item())


Loaded dataset with 57756 images
Train size: 46204 | Val size: 11552
Token-level label counts (train sample up to 200 images): {0: 0, 1: 200}
sample batch shapes: feats torch.Size([16, 1, 900]) labels torch.Size([16, 1]) mask torch.Size([16, 1]) mask.sum: 16


## PARAMATER

In [27]:
# Cell 3: hyperparameters, model definitions, instantiate model, quick forward

# Model hyperparams
EMBED_DIM = 128
NUM_HEADS = 4
NUM_LAYERS = 2
FFN_DIM = 256
DROPOUT = 0.2
LR = 3e-4
WEIGHT_DECAY = 1e-4
LR_SCHED_STEP = 10
LR_SCHED_GAMMA = 0.9
EPOCHS = 20

# Positional encoding (simple sinusoidal)
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        if d_model % 2 == 1:
            # odd dimension case
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        else:
            pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer("pe", pe)

    def forward(self, x):
        # x: (B, L, d_model)
        L = x.size(1)
        return x + self.pe[:, :L, :]

# Transformer detector (per-token logits)
class TransformerDetectorToken(nn.Module):
    def __init__(self, feat_dim, d_model=EMBED_DIM, nhead=NUM_HEADS, num_layers=NUM_LAYERS, dim_feedforward=FFN_DIM, dropout=DROPOUT):
        super().__init__()
        self.input_proj = nn.Linear(feat_dim, d_model)
        self.pos = PositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers)
        self.head = nn.Linear(d_model, 1)   # per-token logit

    def forward(self, x, mask=None):
        # x: (B, L, D)
        z = self.input_proj(x)           # (B, L, d_model)
        z = self.pos(z)
        # src_key_padding_mask: True for padded positions
        src_key_padding_mask = ~mask if mask is not None else None
        out = self.encoder(z, src_key_padding_mask=src_key_padding_mask)  # (B, L, d_model)
        logits = self.head(out).squeeze(-1)   # (B, L)
        return logits

# infer feat_dim from a batch
batch = next(iter(train_loader))
feat_dim = batch[0].shape[-1]
print("Inferred feat_dim:", feat_dim)

# instantiate model, loss, optimizer
detector = TransformerDetectorToken(feat_dim).to(device)
criterion = nn.BCEWithLogitsLoss(reduction='none')
optimizer = optim.AdamW(detector.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=LR_SCHED_STEP, gamma=LR_SCHED_GAMMA)

# quick forward sanity
detector.eval()
with torch.no_grad():
    feats, labels, mask = batch
    feats = feats.to(device); mask = mask.to(device)
    out = detector(feats, mask)
print("detector output shape (B,L):", out.shape)


Inferred feat_dim: 900
detector output shape (B,L): torch.Size([16, 1])


In [28]:
# DEBUG: check label distributions across images
from collections import Counter

label_hist = Counter()
for i in range(min(200, len(dataset))):
    _, labels = dataset[i]
    vals = labels.numpy().astype(int)
    label_hist.update(vals.tolist())

print("Label histogram (0 vs 1):", label_hist)
print("Fraction positive patches:", label_hist[1] / max(1, sum(label_hist.values())))


Label histogram (0 vs 1): Counter({1: 200})
Fraction positive patches: 1.0


In [33]:
from collections import Counter
cnt = Counter()
both = 0
for feats, labels in train_dataset:
    c = Counter(labels.numpy())
    cnt.update(c)
    if 0 in c and 1 in c:
        both += 1
print("Label histogram (0 vs 1):", cnt)
print(f"{both}/{len(train_dataset)} images contain both 0 and 1 labels")


Label histogram (0 vs 1): Counter({1.0: 46204})
0/46204 images contain both 0 and 1 labels


In [18]:
# Cell 4: Training loop with safer masking and validation metric collection

metrics = {"epoch": [], "train_loss": [], "val_loss": [], "val_acc": [], "val_f1": []}

for epoch in range(1, EPOCHS + 1):
    detector.train()
    t0 = time.time()
    train_loss_accum = 0.0
    train_tokens = 0

    for feats, labels, mask in train_loader:
        feats = feats.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        mask = mask.to(device, non_blocking=True)

        # basic asserts
        assert feats.dim() == 3
        if mask.float().sum().item() == 0:
            # skip pathological batch
            print("Warning: encountered batch with zero valid tokens; skipping")
            continue

        optimizer.zero_grad()
        logits = detector(feats, mask)         # (B, L)
        loss_all = criterion(logits, labels)   # (B, L)
        loss_masked = (loss_all * mask.float()).sum() / mask.float().sum()
        if torch.isnan(loss_masked):
            print("NaN loss detected; logging logits/labels stats")
            print("logits min/max:", logits.min().item(), logits.max().item())
            print("labels unique:", torch.unique(labels))
            raise RuntimeError("NaN loss")
        loss_masked.backward()
        optimizer.step()

        train_loss_accum += loss_masked.item() * mask.float().sum().item()
        train_tokens += mask.float().sum().item()

    train_loss = train_loss_accum / max(1, train_tokens)

    # Validation
    detector.eval()
    val_loss_accum = 0.0
    val_tokens = 0
    all_preds = []
    all_trues = []

    with torch.no_grad():
        for feats, labels, mask in val_loader:
            feats = feats.to(device); labels = labels.to(device); mask = mask.to(device)
            logits = detector(feats, mask)               # (B,L)
            loss_all = criterion(logits, labels)
            val_loss_accum += (loss_all * mask.float()).sum().item()
            val_tokens += mask.float().sum().item()

            probs = torch.sigmoid(logits)
            # gather valid tokens per sample explicitly
            for b in range(probs.shape[0]):
                Lb = int(mask[b].sum().item())   # valid length for sample b
                if Lb == 0:
                    continue
                pred_vals = (probs[b, :Lb] > 0.5).long().cpu().numpy().tolist()
                true_vals = labels[b, :Lb].long().cpu().numpy().tolist()
                all_preds.extend(pred_vals)
                all_trues.extend(true_vals)

    val_loss = val_loss_accum / max(1, val_tokens)
    if len(all_trues) == 0:
        print("Warning: validation had zero valid tokens! Setting metrics to zero.")
        val_acc = 0.0; val_f1 = 0.0
    else:
        val_acc = accuracy_score(all_trues, all_preds)
        # if only one class present in val, f1_score can crash; handle gracefully
        try:
            val_f1 = f1_score(all_trues, all_preds, average="binary")
        except ValueError:
            val_f1 = 0.0

    metrics["epoch"].append(epoch)
    metrics["train_loss"].append(train_loss)
    metrics["val_loss"].append(val_loss)
    metrics["val_acc"].append(val_acc)
    metrics["val_f1"].append(val_f1)

    scheduler.step()
    print(f"Epoch {epoch}/{EPOCHS} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f} | time: {time.time()-t0:.1f}s")


Epoch 1/20 | Train Loss: 0.000940 | Val Loss: 0.000011 | Val Acc: 1.0000 | Val F1: 1.0000 | time: 45.9s


KeyboardInterrupt: 

In [ ]:
# Cell 5: Save weights/metrics and plot results

SAVE_PATH = "detector_transformer_fixed.pth"
LOG_PATH = "training_metrics_fixed.csv"

torch.save(detector.state_dict(), SAVE_PATH)
pd.DataFrame(metrics).to_csv(LOG_PATH, index=False)
print("Saved model ->", SAVE_PATH)
print("Saved metrics ->", LOG_PATH)

# quick plots
df = pd.DataFrame(metrics)

plt.figure(figsize=(8,4))
plt.plot(df["epoch"], df["train_loss"], label="Train Loss")
plt.plot(df["epoch"], df["val_loss"], label="Val Loss")
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

plt.figure(figsize=(8,4))
plt.plot(df["epoch"], df["val_acc"] * 100, label="Val Accuracy (%)")
plt.plot(df["epoch"], df["val_f1"] * 100, label="Val F1 (%)")
plt.title("Validation Accuracy & F1")
plt.xlabel("Epoch")
plt.ylabel("Score (%)")
plt.legend()
plt.show()


sample_feats.shape: torch.Size([16, 1, 900]) feat_dim: 900
detector output shape: torch.Size([16, 1])
detector params device: cuda:0


/home/parasite/.pyenv/versions/3.10.13/lib/python3.10/site-packages/torch/nn/modules/transformer.py:502: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at ../aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(
